In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

import pandas as pd
import geopandas as gpd
import plotly.express as px
import plotly.io as pio
from IPython.display import display
from plotly.subplots import make_subplots


In [2]:
dataset = pd.read_csv("/Users/nicolasbour/Desktop/Projets_IA/Projets_certif/bloc_03_Machine_Learning/UBER_PICKUPS/data/dataset.csv")

In [3]:
dataset["date/time"] = pd.to_datetime(dataset["date/time"], format="ISO8601")

In [4]:
sample, reste = train_test_split(
    dataset,
    train_size=50000,
    stratify=dataset["locationid"],
    random_state=42
)

In [5]:
print(sample.shape)
print(reste.shape)

(50000, 8)
(18657457, 8)


In [6]:
print(f"Sample size : {len(sample) / len(dataset) * 100:.2f} %")
print(f"Reste size : {len(reste) / len(dataset) * 100:.2f} %")

Sample size : 0.27 %
Reste size : 99.73 %


In [7]:
sample["date/time"].value_counts().sum()

50000

In [8]:
sample.head()

,date/time,lat_x,lon_x,base,weekday,locationid,borough,zone
12452807,2015-05-07 10:51:00,40.688514,-73.964933,B02598,Jeudi,49,Brooklyn,Clinton Hill
10841777,2015-06-04 09:58:00,40.727944,-73.985214,B02764,Jeudi,79,Manhattan,East Village
10099438,2015-05-18 12:44:00,40.756816,-73.972145,B02764,Lundi,162,Manhattan,Midtown East
6064725,2015-06-18 01:04:00,40.620962,-73.954604,B02764,Jeudi,165,Brooklyn,Midwood
993497,2014-05-04 08:32:00,40.726000,-73.992900,B02682,Dimanche,114,Manhattan,Greenwich Village South


In [9]:
begin = pd.Timestamp("2014-05-01 12:00:00")
end = begin + pd.Timedelta(hours=1)
end2 = begin + pd.Timedelta(days=7)

In [10]:
t1 = dataset[(dataset["date/time"] >= begin) & (dataset["date/time"] < end)].copy()
t2 = dataset[(dataset["date/time"] >= begin) & (dataset["date/time"] < end2)].copy()

In [11]:
print(t1.head())
print(t1.shape)

                 date/time    lat_x    lon_x    base weekday  locationid  \
554893 2014-05-01 12:00:00  40.7823 -73.9807  B02512   Jeudi         239   
554894 2014-05-01 12:00:00  40.7739 -73.9639  B02512   Jeudi         237   
554895 2014-05-01 12:00:00  40.7739 -73.9639  B02512   Jeudi         237   
554896 2014-05-01 12:03:00  40.7596 -73.9776  B02512   Jeudi         161   
554897 2014-05-01 12:03:00  40.7420 -73.9904  B02512   Jeudi         234   

          borough                   zone  
554893  Manhattan  Upper West Side South  
554894  Manhattan  Upper East Side South  
554895  Manhattan  Upper East Side South  
554896  Manhattan         Midtown Center  
554897  Manhattan               Union Sq  
(706, 8)


In [12]:
print(t2.head())
print(t2.shape)

                 date/time    lat_x    lon_x    base weekday  locationid  \
554893 2014-05-01 12:00:00  40.7823 -73.9807  B02512   Jeudi         239   
554894 2014-05-01 12:00:00  40.7739 -73.9639  B02512   Jeudi         237   
554895 2014-05-01 12:00:00  40.7739 -73.9639  B02512   Jeudi         237   
554896 2014-05-01 12:03:00  40.7596 -73.9776  B02512   Jeudi         161   
554897 2014-05-01 12:03:00  40.7420 -73.9904  B02512   Jeudi         234   

          borough                   zone  
554893  Manhattan  Upper West Side South  
554894  Manhattan  Upper East Side South  
554895  Manhattan  Upper East Side South  
554896  Manhattan         Midtown Center  
554897  Manhattan               Union Sq  
(142793, 8)


In [13]:
print("Pickups in the selected hour:", len(t1))

fig_one_hour = px.scatter_map(
    t1,
    lat="lat_x",
    lon="lon_x",
    color="borough",
    animation_frame="date/time",
    center={"lat": 40.73, "lon": -73.97},
    hover_name="zone",
    hover_data={"date/time": True, "lat_x": ":.4f", "lon_x": ":.4f", "weekday": True},
    map_style="carto-positron",
    zoom=10,
    height=600,
    title="Prises en charge sur 1 heure en fonction des boroughs",
)
fig_one_hour.show()

fig_one_hour_static = px.scatter_map(
    t1,
        lat="lat_x",
        lon="lon_x",
        color="borough",
        center={"lat": 40.73, "lon": -73.97},
        hover_name="zone",
        hover_data={"date/time": True, "lat_x": ":.4f", "lon_x": ":.4f", "weekday": True},
        map_style="carto-positron",
        zoom=10,
        height=600,
        title="Prises en charge sur 1 heure en fonction des boroughs",
    )
fig_one_hour_static.show()

Pickups in the selected hour: 706


In [14]:
weekday_distrib = dataset.groupby("weekday").agg(
    pickups=("date/time", "size"),
    days=("date/time", "nunique"),
).reset_index().sort_values("weekday")

In [15]:
weekday_distrib = weekday_distrib.sort_values("pickups", ascending=False)
weekday_distrib.head()

,weekday,pickups,days
5,Samedi,3041187,500431
6,Vendredi,3008172,480312
1,Jeudi,2901178,462849
4,Mercredi,2578562,401348
3,Mardi,2525268,378060


In [16]:
fig = px.bar(
    weekday_distrib,
    x="weekday",
    y="pickups",
    text="pickups",
    title="Nombre de prises en charge par jour de la semaine",
    labels={"weekday": "Jour de la semaine", "pickups": "Nombre de prises en charge"},
    color="weekday",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

In [17]:
print("Pickups in the selected hour:", len(t2))

fig_seven_days = px.scatter_map(
    t2,
    lat="lat_x",
    lon="lon_x",
    color="borough",
    animation_frame="weekday",
    center={"lat": 40.73, "lon": -73.97},
    hover_name="zone",
    hover_data={"date/time": True, "lat_x": ":.4f", "lon_x": ":.4f", "weekday": True},
    map_style="carto-positron",
    zoom=10,
    height=600,
    title="Prises en charge sur 7 jours en fonction des boroughs",
)
fig_seven_days.show()

fig_seven_days_static = px.scatter_map(
    t2,
    lat="lat_x",
    lon="lon_x",
    color="borough",
    center={"lat": 40.73, "lon": -73.97},
    hover_name="zone",
    hover_data={"date/time": True, "lat_x": ":.4f", "lon_x": ":.4f", "weekday": True},
    map_style="carto-positron",
    zoom=10,
    height=600,
    title="Prises en charge sur 7 jours en fonction des boroughs",
)
fig_seven_days_static.show()

Pickups in the selected hour: 142793


Les secteurs de prise en charge ne semble pas affectés significativement par les jours de la semaine. Les jours de la semaine en revanche semble affecter le volume de prise en charge.